In [4]:
# =========================================================
# PROFESSOR-STYLE MULTI-RELATIONSHIP GRAPH (FULL CELL)
#
# Changes you asked:
# ✅ Avoid confusing similar node colors (Slide keeps gray; HAS_SLIDE edge has its own blue)
# ✅ Edges are NEUTRAL by default
# ✅ Color ONLY these categorical rels, each with a different ramp:
#    - HAS_COMPETENCE  -> viridis-like (greens→yellow, no purple)
#    - HAS_GRANULARITY -> flame-like (dark→orange→yellow)
#    - HAS_PHASE       -> cool/teal ramp (teal→cyan→light)
# ✅ No node borders
# ✅ Exports: Neo4j CSV+Cypher + Plotly HTML + PNG + SVG in one run folder
# =========================================================

import json, math, datetime as dt
from pathlib import Path

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import plotly.graph_objects as go

# -----------------------------
# CONFIG
# -----------------------------
CSV_PATH = Path("V3_Erfassungstabelle_Psychotraumatology_Graph_database_modelling_nodes_Datagraphs_WS2526_FILLED.xlsm - trauma_slides_ONE_concept_per_s (3).csv")

OUT_ROOT = Path("prof_style_multi_relationship_graph-CORRECTED")
OUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR = OUT_ROOT / f"run_{dt.datetime.now().strftime('%Y%m%d_%H%M%S')}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

EXPORT_DIR = RUN_DIR / "exports"
PLOTS_DIR = RUN_DIR / "plots"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

NODES_CSV = EXPORT_DIR / "nodes.csv"
RELS_CSV  = EXPORT_DIR / "relationships.csv"
CYPHER    = EXPORT_DIR / "neo4j_import.cypher"

PNG_PATH  = PLOTS_DIR / "graph.png"
SVG_PATH  = PLOTS_DIR / "graph.svg"
HTML_PATH = PLOTS_DIR / "graph.html"

# Optional edges
MAKE_NEXT_EDGES = True
MAKE_CO_TAUGHT_WITH = True  # can be dense if modules have many concepts

# Plot controls
TOP_LABELS = 40
SEED = 42

# Edge styling
EDGE_NEUTRAL = "#B8BCC2"
EDGE_NEUTRAL_ALPHA = 0.18

# Whether to color the 3 categorical rel types with distinct ramps
COLOR_CATEGORICAL_RELS = True

# -----------------------------
# Color palettes (hand-picked ramps; no purple)
# -----------------------------
# Competence (viridis-like but without purple): teal/green -> yellow
COMP_VIRIDIS = ["#1FA187", "#4AC16D", "#A0DA39", "#FDE725"]

# Granularity (flame-like): deep brown -> red/orange -> yellow
GRAN_FLAME = ["#3B0F0F", "#8C1D18", "#D1490F", "#F4A300", "#FFE36E"]

# Phase (cool ramp): deep teal -> cyan -> very light
PHASE_COOL = ["#0B3C49", "#1B6B7A", "#2AA7B8", "#8DE4F0", "#D9FBFF"]

# Relationship type accents (distinct from node colors; helps legibility)
RELTYPE_COLORS = {
    "HAS_SLIDE": "#2F6FB0",      # deep-ish blue (distinct from gray Slide nodes)
    "FOCUSES_ON": "#4B5563",     # graphite
    "NEXT": "#0096A6",           # teal-cyan accent
    "CO_TAUGHT_WITH": "#8A8F98", # muted steel
}

print("✅ RUN_DIR:", RUN_DIR)

# -----------------------------
# 0) Load data
# -----------------------------
if not CSV_PATH.exists():
    raise FileNotFoundError(f"CSV not found: {CSV_PATH}")

df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")

required = {
    "module_code","module_name","slide_page",
    "main_concept","phase_category","Competence_type_guess","granularity_guess"
}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"CSV missing columns: {sorted(missing)}")

df = df.copy()
df["main_concept"] = df["main_concept"].astype(str).str.strip()
df = df[
    df["main_concept"].notna()
    & (df["main_concept"] != "")
    & (df["main_concept"].str.lower() != "concept")
].copy()

# stable slide id (unique per module+slide)
df["slide_id"] = (
    df["module_code"].astype(str).str.strip()
    + "::slide_"
    + df["slide_page"].astype(int).astype(str)
)

# -----------------------------
# 1) Build node tables (typed nodes)
# -----------------------------
modules = (
    df[["module_code","module_name"]]
    .drop_duplicates()
    .rename(columns={"module_code":"id", "module_name":"name"})
)
modules["label"] = "Module"

slides = (
    df[["slide_id","module_code","module_name","slide_page"]]
    .drop_duplicates()
    .rename(columns={"slide_id":"id"})
)
slides["label"] = "Slide"
slides["name"] = slides["module_code"].astype(str) + " / slide " + slides["slide_page"].astype(int).astype(str)

def mode_or_blank(s: pd.Series) -> str:
    m = s.mode()
    return m.iloc[0] if len(m) else ""

concepts = (
    df.groupby("main_concept", as_index=False)
      .agg(
          frequency=("main_concept","size"),
          modules=("module_code", lambda x: int(x.nunique())),
          phase_mode=("phase_category", mode_or_blank),
          competence_mode=("Competence_type_guess", mode_or_blank),
          granularity_mode=("granularity_guess", mode_or_blank),
      )
      .rename(columns={"main_concept":"id"})
)
concepts["label"] = "Concept"
concepts["name"] = concepts["id"]

phases = pd.DataFrame({
    "id": sorted(df["phase_category"].fillna("").astype(str).str.strip().replace("", "Unknown").unique())
})
phases["label"] = "Phase"
phases["name"] = phases["id"]

competences = pd.DataFrame({
    "id": sorted(df["Competence_type_guess"].fillna("").astype(str).str.strip().replace("", "Unknown").unique())
})
competences["label"] = "Competence"
competences["name"] = competences["id"]

granularities = pd.DataFrame({
    "id": sorted(df["granularity_guess"].fillna("").astype(str).str.strip().replace("", "Unknown").unique())
})
granularities["label"] = "Granularity"
granularities["name"] = granularities["id"]

nodes = pd.concat([
    modules[["id","label","name"]],
    slides[["id","label","name"]],
    concepts[["id","label","name","frequency","modules","phase_mode","competence_mode","granularity_mode"]],
    phases[["id","label","name"]],
    competences[["id","label","name"]],
    granularities[["id","label","name"]],
], ignore_index=True)

for col in ["frequency","modules","phase_mode","competence_mode","granularity_mode"]:
    if col not in nodes.columns:
        nodes[col] = ""
nodes = nodes.fillna("")

# -----------------------------
# 2) Build relationships (multi-type)
# -----------------------------
rels = []

# (Module)-[:HAS_SLIDE]->(Slide)
for r in slides.itertuples(index=False):
    rels.append({"source": str(r.module_code), "target": str(r.id), "type": "HAS_SLIDE", "value": ""})

# (Slide)-[:FOCUSES_ON]->(Concept)
slide_concept = df[["slide_id","main_concept"]].drop_duplicates()
for r in slide_concept.itertuples(index=False):
    rels.append({"source": str(r.slide_id), "target": str(r.main_concept), "type": "FOCUSES_ON", "value": ""})

# (Concept)-[:HAS_PHASE]->(Phase)
concept_phase = concepts[["id","phase_mode"]].copy()
concept_phase["phase_mode"] = concept_phase["phase_mode"].astype(str).str.strip().replace("", "Unknown")
for r in concept_phase.itertuples(index=False):
    rels.append({"source": str(r.id), "target": str(r.phase_mode), "type": "HAS_PHASE", "value": str(r.phase_mode)})

# (Concept)-[:HAS_COMPETENCE]->(Competence)
concept_comp = concepts[["id","competence_mode"]].copy()
concept_comp["competence_mode"] = concept_comp["competence_mode"].astype(str).str.strip().replace("", "Unknown")
for r in concept_comp.itertuples(index=False):
    rels.append({"source": str(r.id), "target": str(r.competence_mode), "type": "HAS_COMPETENCE", "value": str(r.competence_mode)})

# (Concept)-[:HAS_GRANULARITY]->(Granularity)
concept_gran = concepts[["id","granularity_mode"]].copy()
concept_gran["granularity_mode"] = concept_gran["granularity_mode"].astype(str).str.strip().replace("", "Unknown")
for r in concept_gran.itertuples(index=False):
    rels.append({"source": str(r.id), "target": str(r.granularity_mode), "type": "HAS_GRANULARITY", "value": str(r.granularity_mode)})

# (Concept)-[:NEXT]->(Concept) within each module
if MAKE_NEXT_EDGES:
    df_seq = df.sort_values(["module_code","slide_page"]).copy()
    for module_code, g in df_seq.groupby("module_code"):
        concepts_ordered = g["main_concept"].tolist()
        for a, b in zip(concepts_ordered, concepts_ordered[1:]):
            if a != b:
                rels.append({"source": str(a), "target": str(b), "type": "NEXT", "value": ""})

# (Concept)-[:CO_TAUGHT_WITH]-(Concept) within module
if MAKE_CO_TAUGHT_WITH:
    for module_code, g in df.groupby("module_code"):
        uniq = [u for u in pd.unique(g["main_concept"]) if isinstance(u, str) and u.strip()]
        if len(uniq) > 120:
            uniq = g["main_concept"].value_counts().head(120).index.tolist()
        for i in range(len(uniq)):
            for j in range(i+1, len(uniq)):
                a, b = uniq[i], uniq[j]
                if a != b:
                    rels.append({"source": str(a), "target": str(b), "type": "CO_TAUGHT_WITH", "value": ""})

rels_df = pd.DataFrame(rels).drop_duplicates().reset_index(drop=True)

# -----------------------------
# 3) Export Neo4j CSV + Cypher (parameter-based)
# -----------------------------
nodes.to_csv(NODES_CSV, index=False, encoding="utf-8-sig")
rels_df.to_csv(RELS_CSV, index=False, encoding="utf-8-sig")

cypher_lines = []
cypher_lines.append("// Neo4j import (parameter-based). Paste into Neo4j Browser.\n")
cypher_lines.append("CREATE CONSTRAINT module_id_unique IF NOT EXISTS FOR (m:Module) REQUIRE m.id IS UNIQUE;")
cypher_lines.append("CREATE CONSTRAINT slide_id_unique  IF NOT EXISTS FOR (s:Slide)  REQUIRE s.id IS UNIQUE;")
cypher_lines.append("CREATE CONSTRAINT concept_id_unique IF NOT EXISTS FOR (c:Concept) REQUIRE c.id IS UNIQUE;")
cypher_lines.append("CREATE CONSTRAINT phase_id_unique IF NOT EXISTS FOR (p:Phase) REQUIRE p.id IS UNIQUE;")
cypher_lines.append("CREATE CONSTRAINT comp_id_unique IF NOT EXISTS FOR (k:Competence) REQUIRE k.id IS UNIQUE;")
cypher_lines.append("CREATE CONSTRAINT gran_id_unique IF NOT EXISTS FOR (g:Granularity) REQUIRE g.id IS UNIQUE;\n")
cypher_lines.append(":param nodes => " + json.dumps(nodes.to_dict(orient="records"), ensure_ascii=False) + ";")
cypher_lines.append(":param rels  => " + json.dumps(rels_df.to_dict(orient="records"), ensure_ascii=False) + ";\n")

cypher_lines.append("""
UNWIND $nodes AS row
CALL {
  WITH row
  WITH row WHERE row.label = "Module"
  MERGE (n:Module {id: row.id})
  SET n.name = row.name
  RETURN 1 AS _
}
CALL {
  WITH row
  WITH row WHERE row.label = "Slide"
  MERGE (n:Slide {id: row.id})
  SET n.name = row.name
  RETURN 1 AS _
}
CALL {
  WITH row
  WITH row WHERE row.label = "Concept"
  MERGE (n:Concept {id: row.id})
  SET n.name = row.name,
      n.frequency = row.frequency,
      n.modules = row.modules,
      n.phase_mode = row.phase_mode,
      n.competence_mode = row.competence_mode,
      n.granularity_mode = row.granularity_mode
  RETURN 1 AS _
}
CALL {
  WITH row
  WITH row WHERE row.label = "Phase"
  MERGE (n:Phase {id: row.id})
  SET n.name = row.name
  RETURN 1 AS _
}
CALL {
  WITH row
  WITH row WHERE row.label = "Competence"
  MERGE (n:Competence {id: row.id})
  SET n.name = row.name
  RETURN 1 AS _
}
CALL {
  WITH row
  WITH row WHERE row.label = "Granularity"
  MERGE (n:Granularity {id: row.id})
  SET n.name = row.name
  RETURN 1 AS _
}
RETURN count(*) AS done;
""".strip())

cypher_lines.append("""
UNWIND $rels AS row
WITH row
MATCH (a {id: row.source})
MATCH (b {id: row.target})
FOREACH (_ IN CASE WHEN row.type = "HAS_SLIDE" THEN [1] ELSE [] END |
  MERGE (a)-[r:HAS_SLIDE]->(b)
  SET r.value = row.value
)
FOREACH (_ IN CASE WHEN row.type = "FOCUSES_ON" THEN [1] ELSE [] END |
  MERGE (a)-[r:FOCUSES_ON]->(b)
  SET r.value = row.value
)
FOREACH (_ IN CASE WHEN row.type = "HAS_PHASE" THEN [1] ELSE [] END |
  MERGE (a)-[r:HAS_PHASE]->(b)
  SET r.value = row.value
)
FOREACH (_ IN CASE WHEN row.type = "HAS_COMPETENCE" THEN [1] ELSE [] END |
  MERGE (a)-[r:HAS_COMPETENCE]->(b)
  SET r.value = row.value
)
FOREACH (_ IN CASE WHEN row.type = "HAS_GRANULARITY" THEN [1] ELSE [] END |
  MERGE (a)-[r:HAS_GRANULARITY]->(b)
  SET r.value = row.value
)
FOREACH (_ IN CASE WHEN row.type = "NEXT" THEN [1] ELSE [] END |
  MERGE (a)-[r:NEXT]->(b)
  SET r.value = row.value
)
FOREACH (_ IN CASE WHEN row.type = "CO_TAUGHT_WITH" THEN [1] ELSE [] END |
  MERGE (a)-[r:CO_TAUGHT_WITH]-(b)
  SET r.value = row.value
);
""".strip())

CYPHER.write_text("\n\n".join(cypher_lines), encoding="utf-8")

print("✅ Exported:", NODES_CSV)
print("✅ Exported:", RELS_CSV)
print("✅ Exported:", CYPHER)

# -----------------------------
# 4) Build graph for plotting
# -----------------------------
G_multi = nx.MultiDiGraph()
for r in nodes.to_dict(orient="records"):
    nid = str(r["id"])
    G_multi.add_node(nid, **r)
for r in rels_df.to_dict(orient="records"):
    G_multi.add_edge(str(r["source"]), str(r["target"]), key=r["type"], **r)

# layout on collapsed simple graph
G_simple = nx.Graph()
G_simple.add_nodes_from(G_multi.nodes(data=True))
for u, v, data in G_multi.edges(data=True):
    G_simple.add_edge(u, v)

pos = nx.spring_layout(
    G_simple,
    seed=SEED,
    k=0.6 / math.sqrt(max(1, G_simple.number_of_nodes()))
)

# -----------------------------
# 5) Node colors by type (NO PURPLE, no oranges that look too similar)
# -----------------------------
NODETYPE_COLORS = {
    "Concept":     "#1F77B4",  # blue
    "Module":      "#2A9D8F",  # teal
    "Slide":       "#6C757D",  # slate
    "Phase":       "#D1495B",  # raspberry (distinct from amber/orange)
    "Competence":  "#E9C46A",  # sand (not orange)
    "Granularity": "#2CA02C",  # green
}
NODE_FALLBACK = "#B0B0B0"

def node_color(n, d):
    lab = (d.get("label","") or "").strip()
    return NODETYPE_COLORS.get(lab, NODE_FALLBACK)

node_colors = [node_color(n, d) for n, d in G_simple.nodes(data=True)]

# -----------------------------
# 6) Edge coloring logic
# -----------------------------
def map_values_to_palette(values, palette):
    vals = []
    for v in values:
        vv = (str(v).strip() if v is not None else "")
        if vv and vv.lower() != "nan":
            vals.append(vv)
    vals = sorted(set(vals))
    if not vals:
        return {}
    out = {}
    for i, v in enumerate(vals):
        out[v] = palette[i % len(palette)]
    return out

phase_values = rels_df.loc[rels_df["type"]=="HAS_PHASE", "value"].tolist()
comp_values  = rels_df.loc[rels_df["type"]=="HAS_COMPETENCE", "value"].tolist()
gran_values  = rels_df.loc[rels_df["type"]=="HAS_GRANULARITY", "value"].tolist()

PHASE_COLOR_MAP = map_values_to_palette(phase_values, PHASE_COOL)
COMP_COLOR_MAP  = map_values_to_palette(comp_values,  COMP_VIRIDIS)
GRAN_COLOR_MAP  = map_values_to_palette(gran_values,  GRAN_FLAME)

def edge_group_key(e):
    et = e.get("type","")
    val = (e.get("value","") or "").strip()

    if COLOR_CATEGORICAL_RELS and et == "HAS_COMPETENCE":
        col = COMP_COLOR_MAP.get(val, EDGE_NEUTRAL)
        return (col, f"HAS_COMPETENCE={val or 'Unknown'}", 0.58)

    if COLOR_CATEGORICAL_RELS and et == "HAS_GRANULARITY":
        col = GRAN_COLOR_MAP.get(val, EDGE_NEUTRAL)
        return (col, f"HAS_GRANULARITY={val or 'Unknown'}", 0.58)

    if COLOR_CATEGORICAL_RELS and et == "HAS_PHASE":
        col = PHASE_COLOR_MAP.get(val, EDGE_NEUTRAL)
        return (col, f"HAS_PHASE={val or 'Unknown'}", 0.58)

    # Non-categorical rels: type-colored but muted (still distinct)
    col = RELTYPE_COLORS.get(et, EDGE_NEUTRAL)
    return (col, et or "REL", EDGE_NEUTRAL_ALPHA)

edge_groups = {}
for u, v, k, data in G_multi.edges(keys=True, data=True):
    col, label, alpha = edge_group_key(data)
    edge_groups.setdefault((col, label, alpha), []).append((u, v, data))

print("✅ Edge groups:", len(edge_groups))

# -----------------------------
# 7) Static plot (PNG/SVG)
# -----------------------------
plt.figure(figsize=(22, 16))

# edges
for (col, label, alpha), items in edge_groups.items():
    edgelist = [(u, v) for u, v, _ in items]
    nx.draw_networkx_edges(
        G_simple, pos,
        edgelist=edgelist,
        edge_color=col,
        width=1.0,
        alpha=float(alpha),
    )

# nodes (no borders)
nx.draw_networkx_nodes(
    G_simple, pos,
    node_size=150,
    node_color=node_colors,
    linewidths=0,
    alpha=0.96
)

# labels: modules + top concepts
freq_map = {str(r["id"]): int(r.get("frequency", 0) or 0) for r in concepts.to_dict(orient="records")}
all_modules = set(modules["id"].astype(str).tolist())
top_concepts = sorted(freq_map.keys(), key=lambda x: freq_map.get(x, 0), reverse=True)[:TOP_LABELS]
labels = {n: n for n in G_simple.nodes() if (n in all_modules or n in top_concepts)}
nx.draw_networkx_labels(G_simple, pos, labels=labels, font_size=8)

# legend: node types
legend_items = [
    plt.Line2D([0],[0], marker='o', color='w', label=k, markerfacecolor=col, markersize=9)
    for k, col in NODETYPE_COLORS.items()
]
plt.legend(handles=legend_items, loc="best", frameon=True)

plt.axis("off")
plt.tight_layout()
plt.savefig(PNG_PATH, dpi=220)
plt.savefig(SVG_PATH, format="svg")
plt.close()

print("✅ Saved:", PNG_PATH)
print("✅ Saved:", SVG_PATH)

# -----------------------------
# 8) Plotly interactive (HTML)
# -----------------------------
plotly_traces = []

# edges: show legend for the 3 categorical ones; also show for HAS_SLIDE / FOCUSES_ON / NEXT / CO_TAUGHT_WITH
def should_show_edge_legend(label: str) -> bool:
    return label.startswith("HAS_") or label in {"HAS_SLIDE","FOCUSES_ON","NEXT","CO_TAUGHT_WITH"}

for (col, label, alpha), items in sorted(edge_groups.items(), key=lambda x: x[0][1]):
    edge_x, edge_y = [], []
    for u, v, data in items:
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]

    plotly_traces.append(
        go.Scatter(
            x=edge_x, y=edge_y,
            mode="lines",
            line=dict(width=1.1, color=col),
            opacity=float(alpha),
            hoverinfo="none",
            name=label,
            showlegend=should_show_edge_legend(label),
        )
    )

# nodes: one trace per node type for legend
nodes_by_type = {}
for n, d in G_simple.nodes(data=True):
    t = (d.get("label","") or "Other").strip()
    nodes_by_type.setdefault(t, []).append((n, d))

for t, items in sorted(nodes_by_type.items(), key=lambda x: x[0]):
    xs, ys, ht = [], [], []
    for n, d in items:
        x, y = pos[n]
        xs.append(x); ys.append(y)
        ht.append("<br>".join([
            f"<b>{n}</b>",
            f"type: {d.get('label','')}",
            f"name: {d.get('name','')}",
        ]))
    plotly_traces.append(
        go.Scatter(
            x=xs, y=ys,
            mode="markers",
            marker=dict(
                size=9,
                color=NODETYPE_COLORS.get(t, NODE_FALLBACK),
                opacity=0.96,
                line=dict(width=0),
            ),
            hovertext=ht,
            hoverinfo="text",
            name=f"Node: {t}",
            showlegend=True,
        )
    )

fig = go.Figure(data=plotly_traces)
fig.update_layout(
    title="Professor-style multi-relationship graph (distinct node types; categorical edges use 3 different ramps)",
    hovermode="closest",
    margin=dict(l=20, r=20, t=60, b=20),
    legend=dict(itemsizing="constant"),
)
fig.write_html(str(HTML_PATH), include_plotlyjs="cdn")
print("✅ Saved:", HTML_PATH)

print("\nDONE ✅")
print("Open this folder:", RUN_DIR)


/home/caroline/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


✅ RUN_DIR: prof_style_multi_relationship_graph-CORRECTED/run_20260212_230646
✅ Exported: prof_style_multi_relationship_graph-CORRECTED/run_20260212_230646/exports/nodes.csv
✅ Exported: prof_style_multi_relationship_graph-CORRECTED/run_20260212_230646/exports/relationships.csv
✅ Exported: prof_style_multi_relationship_graph-CORRECTED/run_20260212_230646/exports/neo4j_import.cypher
✅ Edge groups: 14
✅ Saved: prof_style_multi_relationship_graph-CORRECTED/run_20260212_230646/plots/graph.png
✅ Saved: prof_style_multi_relationship_graph-CORRECTED/run_20260212_230646/plots/graph.svg
✅ Saved: prof_style_multi_relationship_graph-CORRECTED/run_20260212_230646/plots/graph.html

DONE ✅
Open this folder: prof_style_multi_relationship_graph-CORRECTED/run_20260212_230646


In [5]:
# -----------------------------
# Aura-friendly export (split by label + split by rel type)
# -----------------------------
AURA_DIR = OUT_ROOT / "aura_import"
AURA_DIR.mkdir(parents=True, exist_ok=True)

# Nodes (one label per file)
modules[["id","name"]].to_csv(AURA_DIR / "Module.csv", index=False, encoding="utf-8-sig")
slides.rename(columns={"id":"id"})[["id","name"]].to_csv(AURA_DIR / "Slide.csv", index=False, encoding="utf-8-sig")

concepts_out = concepts.rename(columns={"id":"id", "name":"name"})[
    ["id","name","frequency","modules","phase_mode","competence_mode","granularity_mode"]
]
concepts_out.to_csv(AURA_DIR / "Concept.csv", index=False, encoding="utf-8-sig")

phases[["id","name"]].to_csv(AURA_DIR / "Phase.csv", index=False, encoding="utf-8-sig")
competences[["id","name"]].to_csv(AURA_DIR / "Competence.csv", index=False, encoding="utf-8-sig")
granularities[["id","name"]].to_csv(AURA_DIR / "Granularity.csv", index=False, encoding="utf-8-sig")

# Relationships (one type per file)
rels_df = rels_df.copy()
rels_df.rename(columns={"source":"source_id","target":"target_id"}, inplace=True)

def export_rel(rel_type, cols):
    sub = rels_df[rels_df["type"] == rel_type].copy()
    sub = sub[cols]
    sub.to_csv(AURA_DIR / f"{rel_type}.csv", index=False, encoding="utf-8-sig")

export_rel("HAS_SLIDE",        ["source_id","target_id"])
export_rel("FOCUSES_ON",       ["source_id","target_id"])
export_rel("HAS_PHASE",        ["source_id","target_id","value"])
export_rel("HAS_COMPETENCE",   ["source_id","target_id","value"])
export_rel("HAS_GRANULARITY",  ["source_id","target_id","value"])
export_rel("NEXT",             ["source_id","target_id"])
export_rel("CO_TAUGHT_WITH",   ["source_id","target_id"])

print("✅ Aura-friendly CSVs exported to:", AURA_DIR)


✅ Aura-friendly CSVs exported to: prof_style_multi_relationship_graph-CORRECTED/aura_import
